# Week 06 — Python Solution Lab
## Work and Energy

**Companion to `notebooks/Week_06.ipynb`.** This notebook contains *fully worked Python
solutions* to selected problems from that week's problem set — one at **each difficulty level**.

Every solution follows the course's core workflow:

> **Diagram → Principle → Equation → Predict → Verify**

The markdown cell states the problem, identifies the governing principle, and gives the **hand
prediction you should make before running anything**. The code cell then computes the result and
*verifies* it — typically by a second independent method (energy vs. forces, symbolic vs.
numerical, closed form vs. simulation) — and includes `assert` checks against the known answer.

### How to use this notebook

1. **Attempt the problem in `Week_06.ipynb` first.** These solutions are worth very little
   if you read them before trying.
2. Make the hand prediction. Write it down.
3. Run the code cell and compare.
4. **Change a number and re-run.** Every solution is written so that the parameters sit at the
   top; the sweeps and plots update automatically. Ask "what if the mass doubled?" and answer it
   in ten seconds.

### Why the code looks like this

These are not minimal answer-generators. Each one demonstrates something Python does that hand
algebra cannot: parameter sweeps, root-finding, numerical integration, symbolic differentiation,
or a cross-check to machine precision. The physics is the point; the code is how we prove the
physics is right.

---


### Solutions in this notebook

| Level | Problem | Topic | Python technique |
|---|---|---|---|
| **L1 · Basic** | `P4` | Power | peak vs average power from a duty cycle |
| **L2 · Intermediate** | `P8` | Work-Energy with a Variable Force | 4 integration routes: sympy/quad/trapz/geometry |
| **L3 · Challenge** | `P9` | Bungee Jump Analysis | root-finding + full ODE simulation |

---

## L1 · Basic — P4: Power

> **Problem (Week_06.ipynb, L1 — P4).** An elevator motor lifts an $800$ kg car and its
> $200$ kg load a height of $30.0$ m in $45.0$ s at constant speed. What power must the motor
> deliver?

**Diagram → Principle.** Constant speed means zero net force, so the motor's upward force exactly
equals the total weight. Power is the rate of doing work.

**Equation.** $P = W/t = mgh/t$, equivalently $P = Fv$.

**Hand prediction.** $(1000)(9.81)(30.0)/45.0 = 6540$ W $= 6.54$ kW.

**What Python adds.** Two independent formulations — $mgh/t$ and $Fv$ — must give the same
number; computing both is a free correctness check. We then show why *average* power hides the
real design constraint: a real elevator accelerates, and the **peak** power during acceleration
is what sizes the motor. For a trapezoidal profile with a $5$ s ramp,
$v_c = 30/40 = 0.75$ m/s and $a = 0.15$ m/s², so

$$P_{\max} = m(g+a)v_c = 1000(9.96)(0.75) = 7.47\ \text{kW} = 1.142\,P_{\rm avg}.$$

> **Note on method.** We build the acceleration **piecewise and exactly** rather than with
> `np.gradient`. A numerical gradient smears the two step discontinuities across one grid
> spacing and reports a peak a few percent low — an easy way to get a plausible-looking wrong
> answer out of otherwise correct code.

In [ ]:
# ═══ W06 · L1 · P4 — Average power, and why peak power sizes the motor ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
m_car, m_load, h, t_total, g = 800.0, 200.0, 30.0, 45.0, 9.81
m = m_car + m_load

# --- PREDICT (route 1: work over time) ----------------------------------
W = m * g * h
P_avg = W / t_total
print(f"total mass    m = {m:.0f} kg")
print(f"work done     W = m g h   = {W:,.0f} J")
print(f"average power P = W / t   = {P_avg:,.1f} W = {P_avg/1000:.2f} kW")

# --- VERIFY (route 2: P = F v, at constant speed) -----------------------
v = h / t_total
F = m * g                       # constant speed => motor force = weight
P_Fv = F * v
print(f"\nroute 2: v = h/t = {v:.4f} m/s, F = m g = {F:,.0f} N")
print(f"         P = F v = {P_Fv:,.1f} W   -> agrees with route 1")
assert np.isclose(P_avg, P_Fv)
print(f"\nAlso: {P_avg/746:.2f} horsepower.")

# --- The engineering caveat: real lifts accelerate ----------------------
# Trapezoidal speed profile: accelerate 5 s, cruise, decelerate 5 s, same 30 m in 45 s.
t_acc = 5.0
v_cruise = h / (t_total - t_acc)          # area of the trapezoid = h
a = v_cruise / t_acc
# The peak power occurs at the LAST instant of the ramp, where v is already
# v_cruise but a has not yet dropped to zero. That is a one-sided limit at a
# discontinuity, so the time grid has to contain a sample just inside the ramp
# -- a uniform linspace will straddle it and under-report the peak.
eps = 1e-9
t = np.union1d(np.linspace(0, t_total, 2001),
               [t_acc - eps, t_acc, t_total - t_acc, t_total - t_acc + eps])
vel = np.piecewise(t,
        [t < t_acc, (t >= t_acc) & (t <= t_total - t_acc), t > t_total - t_acc],
        [lambda s: a*s, v_cruise, lambda s: v_cruise - a*(s - (t_total - t_acc))])
# Use the EXACT piecewise acceleration. np.gradient would smear the two step
# discontinuities over a grid spacing and quietly under-report the peak.
acc = np.piecewise(t,
        [t < t_acc, (t >= t_acc) & (t <= t_total - t_acc), t > t_total - t_acc],
        [a, 0.0, -a])
P_inst = m * (g + acc) * vel               # motor force = m(g + a), power = F v

# closed form: the peak is the last instant of the ramp, v = v_cruise with a still applied
P_peak = m * (g + a) * v_cruise

print(f"\nWith a realistic trapezoidal profile (5 s ramp up / 5 s ramp down):")
print(f"  cruise speed  = {v_cruise:.3f} m/s   (accel a = {a:.3f} m/s^2)")
print(f"  PEAK power    = m(g+a)v_c = 1000*({g}+{a:.2f})*{v_cruise:.2f} = {P_peak:,.0f} W "
      f"= {P_peak/1000:.2f} kW")
print(f"  cruise power  = m g v_c = {m*g*v_cruise:,.1f} W (once a = 0)")
print(f"  average power = {np.trapezoid(P_inst, t)/t_total/1000:.2f} kW  (matches m g h / t)")
print(f"  -> the motor must be sized for {P_peak/P_avg:.3f}x the average figure.")
assert abs(P_inst.max() - P_peak) < 1e-3, "grid maximum must reach the closed-form peak"
# what a plain uniform grid would have reported, for contrast
t_naive = np.linspace(0, t_total, 2001)
v_naive = np.piecewise(t_naive,
        [t_naive < t_acc, (t_naive >= t_acc) & (t_naive <= t_total - t_acc),
         t_naive > t_total - t_acc],
        [lambda s: a*s, v_cruise, lambda s: v_cruise - a*(s - (t_total - t_acc))])
P_naive = (m*(g + np.gradient(v_naive, t_naive))*v_naive).max()
print(f"  (a uniform grid with np.gradient would have reported {P_naive/1000:.2f} kW --")
print(f"   {100*(1 - P_naive/P_peak):.1f}% low, because it smooths the step in a.)")
assert abs(np.trapezoid(P_inst, t)/t_total - P_avg) / P_avg < 1e-3
assert abs(P_peak - 7470.0) < 1.0 and abs(P_peak/P_avg - 1.1422) < 1e-3

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
ax1.plot(t, vel, color="#1565c0", lw=2); ax1.set_ylabel("v (m/s)")
ax1.set_title("trapezoidal speed profile")
ax2.plot(t, P_inst/1000, color="#e65100", lw=2, label="instantaneous")
ax2.axhline(P_avg/1000, ls="--", c="#2e7d32", label=f"average {P_avg/1000:.2f} kW")
ax2.axhline(P_peak/1000, ls=":", c="crimson", label=f"peak {P_peak/1000:.2f} kW")
ax2.set_ylabel("power (kW)"); ax2.set_title("peak > average"); ax2.legend(fontsize=8)
for ax in (ax1, ax2):
    ax.set_xlabel("t (s)"); ax.grid(alpha=.3)
plt.suptitle("W06 P4 — 1000 kg lifted 30 m in 45 s", y=1.03)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(P_avg - 6540) < 1.0
print(f"[OK] Matches textbook answer: P = {P_avg:,.0f} W = {P_avg/1000:.2f} kW")

## L2 · Intermediate — P8: Work-Energy with a Variable Force

> **Problem (Week_06.ipynb, L2 — P8).** A force $F(x) = 6.0x - 2.0$ N acts on a $2.0$ kg object.
> (a) How much work as it moves from $x = 1.0$ m to $x = 4.0$ m? (b) If it starts at $3.0$ m/s at
> $x = 1.0$ m, what is its speed at $x = 4.0$ m?

**Diagram → Principle.** When $F$ varies with position, work is no longer $Fd$ — it is the **area
under the $F$–$x$ curve**, i.e. an integral. Then the work–energy theorem converts that to speed.

**Equation.** $W = \int_{x_1}^{x_2} F(x)\,dx = [3x^2 - 2x]_1^4$; $\tfrac12mv_2^2 = \tfrac12mv_1^2 + W$.

**Hand prediction.** $W = (48-8) - (3-2) = 39$ J; $v_2 = \sqrt{9 + 39} = 6.93$ m/s.

**What Python adds.** Four routes to the same integral — SymPy exact, `scipy.integrate.quad`,
the trapezoidal rule, and the geometric trapezoid area — all agreeing to machine precision. That
is how you build confidence in a numerical answer when the closed form is unavailable or
hard to trust. We also
integrate the motion forward in $x$ to get the full $v(x)$ curve, not just the endpoint.

In [ ]:
# ═══ W06 · L2 · P8 — A variable force: four routes to the same work integral ═══
import numpy as np
import sympy as sp
from scipy.integrate import quad
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
m, v1, x1, x2 = 2.0, 3.0, 1.0, 4.0
F = lambda x: 6.0 * x - 2.0            # newtons, x in metres

# --- (a) ROUTE 1: SymPy, exact ------------------------------------------
xs = sp.symbols('x', real=True)
W_sym = sp.integrate(6*xs - 2, (xs, x1, x2))
# --- ROUTE 2: scipy adaptive quadrature ---------------------------------
W_quad, err = quad(F, x1, x2)
# --- ROUTE 3: trapezoidal rule on a grid --------------------------------
xg = np.linspace(x1, x2, 200001)
W_trap = np.trapezoid(F(xg), xg)
# --- ROUTE 4: it is a trapezium, so use its area ------------------------
W_geom = 0.5 * (F(x1) + F(x2)) * (x2 - x1)

print("(a) work done from x = 1 m to x = 4 m")
print(f"    sympy (exact)      W = {W_sym} J")
print(f"    scipy quad         W = {W_quad:.10f} J   (est. error {err:.1e})")
print(f"    trapezoid rule     W = {W_trap:.10f} J")
print(f"    geometric area     W = {W_geom:.10f} J")
assert np.allclose([float(W_sym), W_quad, W_trap, W_geom], 39.0)
W = float(W_sym)

# --- (b) work-energy theorem --------------------------------------------
KE1 = 0.5 * m * v1**2
KE2 = KE1 + W
v2  = np.sqrt(2 * KE2 / m)
print(f"\n(b) KE1 = {KE1:.2f} J,  +W = {W:.1f} J,  KE2 = {KE2:.2f} J")
print(f"    v2 = sqrt(2 KE2/m) = {v2:.4f} m/s")

# --- The constant-force trap --------------------------------------------
F_mid = F((x1 + x2)/2)
print(f"\n  Common error: using F at the midpoint ({F_mid:.1f} N) x d = "
      f"{F_mid*(x2-x1):.1f} J")
print(f"  Here it happens to be right (F is linear), but for F(x) = 6x^2 - 2 it would give")
Fq = lambda x: 6*x**2 - 2
print(f"    midpoint rule {Fq((x1+x2)/2)*(x2-x1):.2f} J  vs  true {quad(Fq, x1, x2)[0]:.2f} J")
print("  -> only integrate. Do not sample.")

# --- v(x) for the whole trip, not just the endpoint ---------------------
xx  = np.linspace(x1, x2, 600)
Wx  = np.array([quad(F, x1, xe)[0] for xe in xx])
vx  = np.sqrt(np.maximum(2*(KE1 + Wx)/m, 0))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.7))
ax1.plot(xg, F(xg), color="#1565c0", lw=2)
ax1.fill_between(xg, 0, F(xg), color="#1565c0", alpha=.2)
ax1.axhline(0, c="k", lw=.8)
ax1.set_xlabel("x (m)"); ax1.set_ylabel("F (N)")
ax1.set_title(f"work = shaded area = {W:.0f} J"); ax1.grid(alpha=.3)

ax2.plot(xx, vx, color="#e65100", lw=2)
ax2.plot([x1, x2], [v1, v2], "o", color="crimson", ms=8, zorder=5)
ax2.set_xlabel("x (m)"); ax2.set_ylabel("v (m/s)")
ax2.set_title(f"speed grows {v1:.1f} -> {v2:.2f} m/s"); ax2.grid(alpha=.3)
plt.suptitle("W06 P8 — variable force F(x) = 6x - 2", y=1.03)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(W - 39.0) < 1e-9 and abs(v2 - 6.928) < 1e-3
print(f"[OK] Matches textbook answer: W = {W:.0f} J, v2 = {v2:.2f} m/s")

## L3 · Challenge — P9: Bungee Jump Analysis

> **Problem (Week_06.ipynb, L3 — P9).** A $65$ kg person jumps from a bridge $45$ m above a
> river. The bungee cord has natural length $15$ m and $k = 85$ N/m. (a) Free-fall distance
> before the cord engages? (b) Maximum extension, by energy conservation. (c) Maximum speed, and
> where does it occur?

**Diagram → Principle.** Two regimes. Free fall for the first $15$ m, then gravity *and* a spring
force. Total mechanical energy is conserved throughout (no drag assumed).

**Equation.** With $x$ = extension beyond natural length:
$mg(L + x) = \tfrac12kx^2 \Rightarrow \tfrac12kx^2 - mgx - mgL = 0$.
Maximum speed where net force is zero: $kx = mg$.

**Hand prediction.** $x$ from the quadratic; $v_{\max}$ at $x = mg/k = 7.50$ m below natural length.

**What Python adds.** The energy equation is a quadratic that students routinely solve with the
wrong root sign. We solve it symbolically **and** by root-finding on the energy function, then
**integrate the equation of motion** with `solve_ivp` and confirm the simulated turning point and
peak speed match the energy prediction. Critically, we also check the jump against the $45$ m
bridge height — a safety question the algebra alone never prompts you to ask.

In [ ]:
# ═══ W06 · L3 · P9 — Bungee jump: energy method, root-finding, and a full simulation ═══
import numpy as np
from scipy.optimize import brentq
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
m, H, L0, k, g = 65.0, 45.0, 15.0, 85.0, 9.81

# --- (a) free fall before the cord does anything ------------------------
print(f"(a) the jumper free-falls the cord's natural length: {L0:.1f} m")
v_engage = np.sqrt(2 * g * L0)
print(f"    speed when the cord first goes taut: {v_engage:.2f} m/s")

# --- (b) maximum extension: energy conservation -------------------------
# Taking the lowest point as reference: m g (L0 + x) = 1/2 k x^2
energy = lambda x: 0.5 * k * x**2 - m * g * x - m * g * L0
a_, b_, c_ = 0.5 * k, -m * g, -m * g * L0
x_quad = (-b_ + np.sqrt(b_**2 - 4*a_*c_)) / (2*a_)        # the positive root
x_root = brentq(energy, 0.1, 100.0)
print(f"\n(b) maximum extension")
print(f"    quadratic formula  x = {x_quad:.4f} m")
print(f"    brentq on E(x)     x = {x_root:.4f} m   -> agree")
assert np.isclose(x_quad, x_root)
depth = L0 + x_quad
print(f"    total fall depth   = {L0:.0f} + {x_quad:.2f} = {depth:.2f} m")

# --- SAFETY: does the jumper hit the river? -----------------------------
clearance = H - depth
print(f"    bridge height {H:.0f} m  ->  clearance above the water = {clearance:.2f} m")
if clearance <= 0:
    print("    *** THE JUMPER WOULD HIT THE WATER -- this cord is unsafe. ***")
else:
    print(f"    safe, but only by {clearance:.2f} m.")

# --- (c) maximum speed: net force zero, i.e. k x = m g ------------------
x_vmax = m * g / k
v_max  = np.sqrt(2*g*(L0 + x_vmax) - k*x_vmax**2/m)
print(f"\n(c) net force vanishes at extension x = m g / k = {x_vmax:.4f} m")
print(f"    i.e. {L0 + x_vmax:.2f} m below the bridge")
print(f"    v_max = {v_max:.4f} m/s  ({v_max*3.6:.1f} km/h)")

# --- VERIFY by simulating the actual motion -----------------------------
def rhs(t, y):
    s, v = y                                  # s = distance fallen
    F = m * g - (k * (s - L0) if s > L0 else 0.0)
    return [v, F / m]

sol = solve_ivp(rhs, [0, 12], [0.0, 0.0], max_step=1e-3, rtol=1e-10, atol=1e-12,
                dense_output=True)
ts = np.linspace(0, 12, 20000)
s_t, v_t = sol.sol(ts)
sim_depth = s_t.max()
sim_vmax  = v_t.max()
sim_x_at_vmax = s_t[np.argmax(v_t)] - L0

print(f"\n  simulation cross-check (solve_ivp):")
print(f"    max depth  {sim_depth:.4f} m   vs energy method {depth:.4f} m")
print(f"    max speed  {sim_vmax:.4f} m/s vs energy method {v_max:.4f} m/s")
print(f"    at extension {sim_x_at_vmax:.4f} m vs predicted {x_vmax:.4f} m")
assert abs(sim_depth - depth) < 5e-3
assert abs(sim_vmax - v_max) < 5e-3
assert abs(sim_x_at_vmax - x_vmax) < 2e-2

# --- Plot ---------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 4))
ax1.plot(ts, H - s_t, color="#1565c0", lw=2)
ax1.axhline(H - L0, ls=":", c="#2e7d32", label="cord goes taut")
ax1.axhline(H - depth, ls="--", c="#e65100", label=f"lowest point ({H-depth:.1f} m)")
ax1.axhline(0, c="crimson", lw=2, label="river")
ax1.set_xlabel("t (s)"); ax1.set_ylabel("height above river (m)")
ax1.set_title("the jump"); ax1.grid(alpha=.3); ax1.legend(fontsize=8)

xg = np.linspace(0, x_quad*1.05, 400)
ax2.plot(L0 + xg, m*g*(L0 + xg), color="#2e7d32", lw=2, label="energy released by gravity")
ax2.plot(L0 + xg, 0.5*k*xg**2,   color="#e65100", lw=2, label="energy stored in the cord")
ax2.plot(L0 + xg, m*g*(L0+xg) - 0.5*k*xg**2, color="#1565c0", lw=2, label="kinetic energy")
ax2.axvline(L0 + x_vmax, ls=":", c="grey", label="peak speed")
ax2.axvline(depth, ls="--", c="crimson", label="turning point (KE = 0)")
ax2.set_xlabel("distance fallen (m)"); ax2.set_ylabel("energy (J)")
ax2.set_title("the energy budget"); ax2.grid(alpha=.3); ax2.legend(fontsize=8)
plt.suptitle("W06 P9 — 65 kg jumper, 15 m cord, k = 85 N/m", y=1.03)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(x_quad - 24.2746) < 0.01, f"x = {x_quad}"
assert abs(x_vmax - 7.502) < 0.01
assert clearance > 0, "the jumper must not reach the water"
print(f"\n[OK] extension {x_quad:.2f} m, total fall {depth:.2f} m, "
      f"v_max {v_max:.2f} m/s at {L0 + x_vmax:.2f} m -- confirmed by simulation.")

---

## Self-check

**Every code cell above contains one or more `assert` checks.** If you run the whole notebook top
to bottom without an `AssertionError`, all of the numerical checks on this page have passed.
(The asserts sit just before each cell's closing summary, so the last thing you see is a printed
result — not the check itself.)

**Now transfer the skill.** Pick one unsolved problem from `Week_06.ipynb` at the level you
found hardest, and write the same five-part structure for it:

```python
# --- MODEL:   parameters at the top, with units in comments
# --- PREDICT: the closed-form answer
# --- VERIFY:  a SECOND, independent route to the same number
# --- CHECK:   assert against your hand prediction
```

The verify step is the one that matters. A result you have only computed one way is a result you
have not checked.
